# Glossier Product Scraping

## Tasks 1–2: Product Information and Pagination

In [ ]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timezone
import json
import time
import re
import pandas as pd


BASE_URL = "https://www.glossier.com"
COLLECTION_URL = f"{BASE_URL}/collections/all"

headers = {
    "User-Agent": "Mozilla/5.0"
}

products = []
seen_product_ids = set()

page = 1

while True:
    print(f"Scraping page {page}...")

    url = f"{COLLECTION_URL}?page={page}"

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    product_data = None

    # Surandame JSON su produktų duomenimis
    for script in soup.find_all("script"):
        content = script.string or ""

        if '"currencyCode"' in content and '"items"' in content:
            try:
                data = json.loads(content)
                items = data.get("items", [])

                if items and "productId" in items[0]:
                    product_data = data
                    break

            except json.JSONDecodeError:
                continue

    # Jeigu produktų nebėra – baigiame
    if not product_data:
        print("No more products.")
        break

    items = product_data["items"]

    new_products = 0

    for item in items:
        product_id = str(item["productId"])

        # Apsauga nuo pasikartojimų
        if product_id in seen_product_ids:
            continue

        seen_product_ids.add(product_id)

        scraped_at = datetime.now(timezone.utc).isoformat()

        image_url = item.get("image")

        if image_url and image_url.startswith("//"):
            image_url = "https:" + image_url

        product = {
            "product_name": str(item["name"]),
            "product_id": product_id,
            "image": [image_url] if image_url else [],
            "url": f"{BASE_URL}/products/{item['handle']}",
            "price": str(item["price"]),
            "scraped_at": scraped_at
        }

        products.append(product)
        new_products += 1

    print(f"Found {len(items)} items")
    print(f"New products: {new_products}")
    print(f"Total unique products: {len(products)}")
    print()

    # Jei visas puslapis jau buvo matytas – stop
    if new_products == 0:
        print("No new products found. Stopping.")
        break

    page += 1

Scraping page 1...
Found 18 items
New products: 18
Total unique products: 18

Scraping page 2...
Found 18 items
New products: 18
Total unique products: 36

Scraping page 3...
Found 18 items
New products: 18
Total unique products: 54

Scraping page 4...
Found 18 items
New products: 18
Total unique products: 72

Scraping page 5...
Found 18 items
New products: 18
Total unique products: 90

Scraping page 6...
Found 7 items
New products: 7
Total unique products: 97

Scraping page 7...
No more products.


In [2]:
with open("all_products.json", "w", encoding="utf-8") as f:
    json.dump(
        products,
        f,
        ensure_ascii=False,
        indent=4
    )

print(f"Saved {len(products)} products to all_products.json")

Saved 97 products to all_products.json


## Task 3: Product Descriptions

In [3]:
for i, product in enumerate(products, start=1):

    print(f"[{i}/{len(products)}] Scraping: {product['product_name']}")

    try:
        response = requests.get(
            product["url"],
            headers=headers,
            timeout=15
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        description_meta = soup.find(
            "meta",
            attrs={"name": "description"}
        )

        if description_meta:
            product["description"] = description_meta.get(
                "content",
                ""
            ).strip()
        else:
            product["description"] = ""

    except requests.RequestException as e:
        print(f"Error: {e}")
        product["description"] = ""

    # Nedidelė pauzė tarp requestų
    time.sleep(0.5)

[1/97] Scraping: Mini Glossier You Wardrobe Kit
[2/97] Scraping: Glossier You Duo
[3/97] Scraping: Glossier You Doux
[4/97] Scraping: Glossier You
[5/97] Scraping: Boy Brow
[6/97] Scraping: Balm Dotcom
[7/97] Scraping: Fragrance Duo
[8/97] Scraping: In A New York Minute Set
[9/97] Scraping: Travel Spray + Balm
[10/97] Scraping: Fragrance Two Ways
[11/97] Scraping: Lip Line
[12/97] Scraping: Cloud Paint Plush Blush
[13/97] Scraping: Cloud Paint
[14/97] Scraping: The Glossier Icons
[15/97] Scraping: Fragrance Trio
[16/97] Scraping: The Skincare Icons
[17/97] Scraping: Futuredew
[18/97] Scraping: Glossier You Rêve
[19/97] Scraping: Glossier You Soie
[20/97] Scraping: Glossier You Fleur
[21/97] Scraping: Glossier You Discovery Kit
[22/97] Scraping: Embroidered Pink Hoodie
[23/97] Scraping: Lash Slick
[24/97] Scraping: Ultralip
[25/97] Scraping: The Face Kit
[26/97] Scraping: Boy Brow Duo
[27/97] Scraping: Balm Dotcom Trio
[28/97] Scraping: Milky Jelly Cleanser
[29/97] Scraping: Invisible S

In [4]:
with_description = sum(
    1 for product in products
    if product.get("description")
)

without_description = sum(
    1 for product in products
    if not product.get("description")
)

print("Total:", len(products))
print("With description:", with_description)
print("Without description:", without_description)

Total: 97
With description: 88
Without description: 9


In [5]:
with open(
    "all_products_with_descriptions.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        products,
        f,
        ensure_ascii=False,
        indent=4
    )

print("Saved!")

Saved!


In [6]:
missing_products = [
    product
    for product in products
    if not product.get("description")
]

print(f"Missing descriptions: {len(missing_products)}")

for product in missing_products:
    print(product["product_name"])
    print(product["url"])
    print()

Missing descriptions: 9
In A New York Minute Set
https://www.glossier.com/products/in-a-new-york-minute

The Glossier Icons
https://www.glossier.com/products/the-glossier-icons

Boy Brow Duo
https://www.glossier.com/products/boy-brow-duo

You Look Good Pet Set
https://www.glossier.com/products/you-look-good-pet-set

The Curbside Set
https://www.glossier.com/products/the-curbside-set

Birthday Cake Candle
https://www.glossier.com/products/birthday-cake-candle

Care Package
https://www.glossier.com/products/care-package

The Brow Kit
https://www.glossier.com/products/brow-kit

Digital Gift Card
https://www.glossier.com/products/gift-card



In [7]:
missing_products = [
    product
    for product in products
    if not product.get("description")
]

print(f"Retrying {len(missing_products)} products...\n")

for i, product in enumerate(missing_products, start=1):

    print(
        f"[{i}/{len(missing_products)}] "
        f"{product['product_name']}"
    )

    try:
        response = requests.get(
            product["url"],
            headers=headers,
            timeout=15
        )
        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        description_html = soup.select_one(
            "#description-item"
        )

        if description_html:
            product["description"] = (
                description_html.get_text(
                    " ",
                    strip=True
                )
            )

            print("  Description found")
        else:
            print("  Description still missing")

    except requests.RequestException as e:
        print(f"  Error: {e}")

    time.sleep(0.5)

Retrying 9 products...

[1/9] In A New York Minute Set
  Description found
[2/9] The Glossier Icons
  Description found
[3/9] Boy Brow Duo
  Description found
[4/9] You Look Good Pet Set
  Description found
[5/9] The Curbside Set
  Description found
[6/9] Birthday Cake Candle
  Description found
[7/9] Care Package
  Description found
[8/9] The Brow Kit
  Description found
[9/9] Digital Gift Card
  Description still missing


In [8]:
with_description = sum(
    1 for product in products
    if product.get("description")
)

without_description = sum(
    1 for product in products
    if not product.get("description")
)

print()
print("Total:", len(products))
print("With description:", with_description)
print("Without description:", without_description)


Total: 97
With description: 96
Without description: 1


In [9]:
with open(
    "all_products_with_descriptions.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        products,
        f,
        ensure_ascii=False,
        indent=4
    )

## Stretch Task 4: Product Variants

In [10]:
variant_products = []
seen_variant_ids = set()

for i, product in enumerate(products, start=1):

    print(f"[{i}/{len(products)}] {product['product_name']}")

    try:
        response = requests.get(
            product["url"],
            headers=headers,
            timeout=15
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        product_data = None

        # Ieškome Shopify Analytics produkto JSON
        for script in soup.find_all("script"):
            content = script.string or ""

            if "window.ShopifyAnalytics.meta" in content and "var meta =" in content:

                match = re.search(
                    r'var meta = ({.*?});',
                    content,
                    re.DOTALL
                )

                if match:
                    meta_data = json.loads(match.group(1))
                    product_data = meta_data.get("product")
                    break

        if not product_data:
            print("  Product data not found")
            continue

        variants = product_data.get("variants", [])

        print(f"  Variants: {len(variants)}")

        for variant in variants:

            variant_id = str(variant["id"])

            # Apsauga nuo pasikartojančių variantų
            if variant_id in seen_variant_ids:
                continue

            seen_variant_ids.add(variant_id)

            variant_product = {
                "product_name": variant["name"],
                "product_id": variant_id,
                "parent_product_id": str(product_data["id"]),
                "variant": variant.get("public_title"),
                "sku": variant.get("sku"),
                "image": product.get("image", []),
                "url": f"{product['url']}?variant={variant_id}",
                "price": str(variant["price"] / 100),
                "description": product.get("description", ""),
                "scraped_at": datetime.now(timezone.utc).isoformat()
            }

            variant_products.append(variant_product)

    except (requests.RequestException, json.JSONDecodeError) as e:
        print(f"  Error: {e}")

    # Nedidelė pauzė tarp užklausų
    time.sleep(0.5)

print("\nTOTAL UNIQUE VARIANTS:", len(variant_products))


[1/97] Mini Glossier You Wardrobe Kit
  Variants: 1
[2/97] Glossier You Duo
  Variants: 1
[3/97] Glossier You Doux
  Variants: 3
[4/97] Glossier You
  Variants: 3
[5/97] Boy Brow
  Variants: 7
[6/97] Balm Dotcom
  Variants: 12
[7/97] Fragrance Duo
  Variants: 1
[8/97] In A New York Minute Set
  Variants: 1
[9/97] Travel Spray + Balm
  Variants: 1
[10/97] Fragrance Two Ways
  Variants: 1
[11/97] Lip Line
  Variants: 9
[12/97] Cloud Paint Plush Blush
  Variants: 10
[13/97] Cloud Paint
  Variants: 15
[14/97] The Glossier Icons
  Variants: 1
[15/97] Fragrance Trio
  Variants: 1
[16/97] The Skincare Icons
  Variants: 1
[17/97] Futuredew
  Variants: 1
[18/97] Glossier You Rêve
  Variants: 2
[19/97] Glossier You Soie
  Variants: 2
[20/97] Glossier You Fleur
  Variants: 2
[21/97] Glossier You Discovery Kit
  Variants: 1
[22/97] Embroidered Pink Hoodie
  Variants: 7
[23/97] Lash Slick
  Variants: 2
[24/97] Ultralip
  Variants: 9
[25/97] The Face Kit
  Variants: 1
[26/97] Boy Brow Duo
  Variants

In [11]:
print()
print("Original products:", len(products))
print("Variant products:", len(variant_products))
print()

for product in variant_products[:10]:
    print(
        product["product_name"],
        "|",
        product["product_id"],
        "|",
        product["variant"],
        "|",
        product["price"]
    )


Original products: 97
Variant products: 330

Mini Glossier You Wardrobe Kit | 48774149112053 | None | 64.0
Glossier You Duo | 48774147571957 | None | 90.0
Glossier You Doux - 100 ml | 46964207026421 | 100 ml | 116.0
Glossier You Doux - 50 ml | 46147023896821 | 50 ml | 82.0
Glossier You Doux - 8 ml | 46147023929589 | 8 ml | 32.0
Glossier You - 100 ml | 44056890573045 | 100 ml | 116.0
Glossier You - 50 ml | 44056890540277 | 50 ml | 82.0
Glossier You - 8 ml | 45133430554869 | 8 ml | 32.0
Boy Brow - Dark Brown | 43886803190005 | Dark Brown | 22.0
Boy Brow - Brown | 43878814712053 | Brown | 22.0


### Variant Validation

In [12]:
variant_ids = [
    product["product_id"]
    for product in variant_products
]

print("Total variants:", len(variant_ids))
print("Unique variant IDs:", len(set(variant_ids)))
print("Duplicates:", len(variant_ids) - len(set(variant_ids)))

Total variants: 330
Unique variant IDs: 330
Duplicates: 0


In [13]:
with open(
    "all_product_variants.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        variant_products,
        f,
        ensure_ascii=False,
        indent=4
    )

print(
    f"Saved {len(variant_products)} variants "
    "to all_product_variants.json"
)

Saved 330 variants to all_product_variants.json


## Stretch Task 5: US Localization

In [ ]:
BASE_URL = "https://www.glossier.com"

headers = {
    "User-Agent": "Mozilla/5.0"
}

session = requests.Session()
session.headers.update(headers)

# Pirmiausia atidarome svetainę
response = session.get(BASE_URL)
response.raise_for_status()

print("Initial status:", response.status_code)
print("Initial cookies:", session.cookies.get_dict())

Initial status: 200
Initial cookies: {'localization': 'US', 'cart_currency': 'USD', '_shopify_essential': ':AaEhYSJ8AAEA2jfwvGa_vaYxaM_0MjoKFSRIkNl9aOau4MV_WpUbbPbiZdEo6ZcXh0Mpki6Cf2KGw-TM3_3EtJymb7U:', '_shopify_analytics': ':AaEhYSNzAAEAtovxapSteGFpI_Pegs_SgkKg3iXPgesgSyXitFPAw4_7rFXLZhMZSjZGQ089_CQXUY3g8pQIzxVOsMs:', '_shopify_marketing': ':AaEhYSNzAAEAdX8pyXVO658LLRjMlCXp9tlzSRBTBlGQUnfpTOooGSR_7VLKPLQ6mVRNzSC8lxwakv4wohXhJos8IGU:'}


In [15]:
localization_url = f"{BASE_URL}/localization"

data = {
    "country_code": "US",
    "language_code": "en",
    "return_to": "/collections/all"
}

response = session.post(
    localization_url,
    data=data,
    allow_redirects=True
)

print("Localization status:", response.status_code)
print("Final URL:", response.url)
print("Cookies:", session.cookies.get_dict())

Localization status: 404
Final URL: https://www.glossier.com/localization
Cookies: {'localization': 'US', 'cart_currency': 'USD', '_shopify_essential': ':AaEhYSJ8AAEA2jfwvGa_vaYxaM_0MjoKFSRIkNl9aOau4MV_WpUbbPbiZdEo6ZcXh0Mpki6Cf2KGw-TM3_3EtJymb7U:', '_shopify_analytics': ':AaEhYSNzAAEAtovxapSteGFpI_Pegs_SgkKg3iXPgesgSyXitFPAw4_7rFXLZhMZSjZGQ089_CQXUY3g8pQIzxVOsMs:', '_shopify_marketing': ':AaEhYSNzAAEAdX8pyXVO658LLRjMlCXp9tlzSRBTBlGQUnfpTOooGSR_7VLKPLQ6mVRNzSC8lxwakv4wohXhJos8IGU:'}


In [16]:
response = session.get(
    "https://www.glossier.com/collections/all"
)

response.raise_for_status()

html = response.text

print("Status:", response.status_code)
print("USD:", '"currencyCode":"USD"' in html)
print("US country:", 'afterpay_js_country = "US"' in html)
print("Cookies:", session.cookies.get_dict())

Status: 200
USD: True
US country: False
Cookies: {'localization': 'US', 'cart_currency': 'USD', '_shopify_essential': ':AaEhYSJ8AAEA2jfwvGa_vaYxaM_0MjoKFSRIkNl9aOau4MV_WpUbbPbiZdEo6ZcXh0Mpki6Cf2KGw-TM3_3EtJymb7U:', '_shopify_analytics': ':AaEhYSNzAAEAtovxapSteGFpI_Pegs_SgkKg3iXPgesgSyXitFPAw4_7rFXLZhMZSjZGQ089_CQXUY3g8pQIzxVOsMs:', '_shopify_marketing': ':AaEhYSNzAAEAdX8pyXVO658LLRjMlCXp9tlzSRBTBlGQUnfpTOooGSR_7VLKPLQ6mVRNzSC8lxwakv4wohXhJos8IGU:'}


In [17]:
test_url = "https://www.glossier.com/products/boy-brow"

response = session.get(test_url)
response.raise_for_status()

html = response.text

print("Status:", response.status_code)
print("US country:", 'afterpay_js_country = "US"' in html)
print("USD currency:", 'afterpay_shop_currency = "USD"' in html)
print("Localization cookie:", session.cookies.get("localization"))
print("Currency cookie:", session.cookies.get("cart_currency"))

Status: 200
US country: True
USD currency: True
Localization cookie: US
Currency cookie: USD


In [18]:
us_products = []
seen_product_ids = set()

page = 1

while True:
    print(f"Scraping US page {page}...")

    url = f"{BASE_URL}/collections/all?page={page}"

    response = session.get(
        url,
        timeout=15
    )
    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    product_data = None

    for script in soup.find_all("script"):
        content = script.string or ""

        if '"currencyCode"' in content and '"items"' in content:
            try:
                data = json.loads(content)
                items = data.get("items", [])

                if items and "productId" in items[0]:
                    product_data = data
                    break

            except json.JSONDecodeError:
                continue

    if not product_data:
        print("No more products.")
        break

    items = product_data["items"]
    new_products = 0

    for item in items:
        product_id = str(item["productId"])

        if product_id in seen_product_ids:
            continue

        seen_product_ids.add(product_id)

        image_url = item.get("image")

        if image_url and image_url.startswith("//"):
            image_url = "https:" + image_url

        product = {
            "product_name": str(item["name"]),
            "product_id": product_id,
            "image": [image_url] if image_url else [],
            "url": f"{BASE_URL}/products/{item['handle']}",
            "price": str(item["price"]),
            "scraped_at": datetime.now(
                timezone.utc
            ).isoformat()
        }

        us_products.append(product)
        new_products += 1

    print(f"Found {len(items)} items")
    print(f"New products: {new_products}")
    print(f"Total US products: {len(us_products)}")
    print()

    if new_products == 0:
        break

    page += 1

Scraping US page 1...
Found 18 items
New products: 18
Total US products: 18

Scraping US page 2...
Found 18 items
New products: 18
Total US products: 36

Scraping US page 3...
Found 18 items
New products: 18
Total US products: 54

Scraping US page 4...
Found 18 items
New products: 18
Total US products: 72

Scraping US page 5...
Found 18 items
New products: 18
Total US products: 90

Scraping US page 6...
Found 7 items
New products: 7
Total US products: 97

Scraping US page 7...
No more products.


In [19]:
print("US products:", len(us_products))

for product in us_products[:5]:
    print(
        product["product_name"],
        "|",
        product["price"]
    )

US products: 97
Mini Glossier You Wardrobe Kit | 64.0
Glossier You Duo | 90.0
Glossier You Doux | 32.0
Glossier You | 32.0
Boy Brow | 22.0


In [20]:
with open(
    "us_products.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        us_products,
        f,
        ensure_ascii=False,
        indent=4
    )

print(f"Saved {len(us_products)} US products to us_products.json")

Saved 97 US products to us_products.json


In [21]:

for i, product in enumerate(us_products, start=1):

    print(f"[{i}/{len(us_products)}] {product['product_name']}")

    try:
        response = session.get(
            product["url"],
            timeout=15
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        description_meta = soup.find(
            "meta",
            attrs={"name": "description"}
        )

        if description_meta:
            product["description"] = description_meta.get(
                "content", ""
            ).strip()
        else:
            product["description"] = ""

    except requests.RequestException as e:
        print(f"Error: {e}")
        product["description"] = ""

    time.sleep(0.5)

with_description = sum(
    1 for product in us_products
    if product.get("description")
)

print("\nTOTAL US PRODUCTS:", len(us_products))
print("WITH DESCRIPTION:", with_description)
print("WITHOUT DESCRIPTION:", len(us_products) - with_description)


[1/97] Mini Glossier You Wardrobe Kit
[2/97] Glossier You Duo
[3/97] Glossier You Doux
[4/97] Glossier You
[5/97] Boy Brow
[6/97] Balm Dotcom
[7/97] Fragrance Duo
[8/97] In A New York Minute Set
[9/97] Travel Spray + Balm
[10/97] Fragrance Two Ways
[11/97] Lip Line
[12/97] Cloud Paint Plush Blush
[13/97] Cloud Paint
[14/97] The Glossier Icons
[15/97] Fragrance Trio
[16/97] The Skincare Icons
[17/97] Futuredew
[18/97] Glossier You Rêve
[19/97] Glossier You Soie
[20/97] Glossier You Fleur
[21/97] Glossier You Discovery Kit
[22/97] Embroidered Pink Hoodie
[23/97] Lash Slick
[24/97] Ultralip
[25/97] The Face Kit
[26/97] Boy Brow Duo
[27/97] Balm Dotcom Trio
[28/97] Milky Jelly Cleanser
[29/97] Invisible Shield SPF50
[30/97] Perfecting Skin Tint
[31/97] Stretch Balm Concealer
[32/97] Stretch Fluid Foundation
[33/97] Crème de You
[34/97] Mini Beauty Bag
[35/97] The Beauty Bag
[36/97] The No-Makeup Makeup Kit
[37/97] Errand Bag
[38/97] Lip Gloss
[39/97] Boy Brow Arch
[40/97] Hand Cream
[41/97

In [22]:


missing_products = [
    product for product in us_products
    if not product.get("description")
]

print("Retrying:", len(missing_products), "products")

for i, product in enumerate(missing_products, start=1):
    print(f"[{i}/{len(missing_products)}] {product['product_name']}")

    try:
        response = session.get(product["url"], timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        description_meta = soup.find(
            "meta",
            attrs={"name": "description"}
        )

        product["description"] = (
            description_meta.get("content", "").strip()
            if description_meta else ""
        )

    except requests.RequestException as e:
        print("Error:", e)

    time.sleep(1)

print("\nTotal US products:", len(us_products))
print("With description:", sum(bool(p.get("description")) for p in us_products))
print("Without description:", sum(not p.get("description") for p in us_products))


Retrying: 9 products
[1/9] In A New York Minute Set
[2/9] The Glossier Icons
[3/9] Boy Brow Duo
[4/9] You Look Good Pet Set
[5/9] The Curbside Set
[6/9] Birthday Cake Candle
[7/9] Care Package
[8/9] The Brow Kit
[9/9] Digital Gift Card

Total US products: 97
With description: 88
Without description: 9


In [23]:

for product in us_products:
    if product.get("description"):
        continue

    try:
        response = session.get(product["url"], timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        og_meta = soup.find(
            "meta",
            attrs={"property": "og:description"}
        )

        if og_meta:
            product["description"] = og_meta.get(
                "content", ""
            ).strip()

        print(
            product["product_name"],
            "| Description found:",
            bool(product.get("description"))
        )

    except requests.RequestException as e:
        print("Error:", product["product_name"], e)

    time.sleep(0.5)

print("\nTotal US products:", len(us_products))
print(
    "With description:",
    sum(bool(p.get("description")) for p in us_products)
)
print(
    "Without description:",
    sum(not p.get("description") for p in us_products)
)


In A New York Minute Set | Description found: True
The Glossier Icons | Description found: True
Boy Brow Duo | Description found: True
You Look Good Pet Set | Description found: True
The Curbside Set | Description found: True
Birthday Cake Candle | Description found: True
Care Package | Description found: True
The Brow Kit | Description found: True
Digital Gift Card | Description found: True

Total US products: 97
With description: 97
Without description: 0


In [24]:
print("product_name:", type(us_products[0]["product_name"]))
print("product_id:", type(us_products[0]["product_id"]))
print("image:", type(us_products[0]["image"]))
print("url:", type(us_products[0]["url"]))
print("price:", type(us_products[0]["price"]))
print("scraped_at:", type(us_products[0]["scraped_at"]))

product_name: <class 'str'>
product_id: <class 'str'>
image: <class 'list'>
url: <class 'str'>
price: <class 'str'>
scraped_at: <class 'str'>


In [ ]:
with open("all_products_with_descriptions.json", "r", encoding="utf-8") as f:
    products = json.load(f)

pd.DataFrame(products).to_csv(
    "glossier_products.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved: glossier_products.csv ({len(products)} products)")

Saved: glossier_products.csv (97 products)
